In [1]:
from modeling_amt.language_modeling import AssociativeRecurrentWrapper as RecurrentWrapper, AssociativeMemoryCell as MemoryCell
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B', attn_implementation="eager")
model = RecurrentWrapper(
    segment_size=5, 
    sliding_window=True,
    memory_cell=MemoryCell(base_model=base_model, num_mem_tokens=4, d_mem=4, layers_attr='model.layers')
)

In [5]:
import torch
input_ids = torch.arange(1, 13).unsqueeze(0).long()
# input_ids = torch.ones(1, 7).long()
attention_mask = torch.ones(1, 12).long()
input_ids.shape

torch.Size([1, 12])

In [6]:

generate_ids = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=3, eos_token_id=100)
print(generate_ids.shape)

tensor([[11.9833,  7.3937,  9.4880,  6.3985, 10.5126]])
tensor([[15.6303,  8.0598, 11.1294, 10.1811, 13.1236]])
tensor([[12.4585,  6.6385, 11.1940,  8.7731, 13.0842]])
tensor([[12.4266,  8.8000, 12.6414, 10.1559, 13.0527]])
tensor([[11.2053,  9.9485, 12.0190, 11.0248, 11.1886]])
torch.Size([1, 3])


In [7]:
fwd_logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
fwd_logits[:, -1, :5]

tensor([[15.6302,  8.0598, 11.1294, 10.1811, 13.1236]],
       grad_fn=<SliceBackward0>)

In [7]:
input_ids = torch.cat([input_ids, generate_ids[:, :1]], dim=-1)
attention_mask = torch.cat([attention_mask, torch.ones(1, 1)], dim=-1)
input_ids.shape

torch.Size([1, 8])

In [8]:
fwd_logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
fwd_logits[:, -1, :5]

tensor([[15.6324, 15.5902, 16.5040, 20.9826, 16.9247]],
       grad_fn=<SliceBackward0>)

In [7]:
generate_ids

tensor([[ 11, 323, 279]])

[{'input_ids': tensor([[1, 2, 3, 4, 5]]),
  'attention_mask': tensor([[1, 1, 1, 1, 1]])},
 {'input_ids': tensor([[6, 7]]), 'attention_mask': tensor([[1, 1]])}]

[{'input_ids': tensor([[1, 2, 3, 4, 5]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}, {'input_ids': tensor([[6, 7]]), 'attention_mask': tensor([[1, 1]])}]


In [71]:
segs = model.segment(input_ids=input_ids, attention_mask=attention_mask)
cache = base_model(**segs[0], use_cache=True).past_key_values

attn0 = model.attn_mask_to_4d(attn_mask=segs[0]['attention_mask'], upper=True, query_len=2)
attn1 = model.attn_mask_to_4d(attn_mask=segs[1]['attention_mask'], upper=False, query_len=2)
# attn0[0][0][-1][0] = 1
segs[1]['attention_mask'] = torch.cat([attn0, attn1], dim=-1)
segs[1]['attention_mask'] = model.memory_cell.convert_to_infinity_attn_mask(segs[1]['attention_mask'], dtype=torch.bfloat16)
out = base_model(input_ids=segs[1]['input_ids'][:, :1], attention_mask=segs[1]['attention_mask'][..., :1, :-1], past_key_values=cache, use_cache=True)
cache = out.past_key_values
print(out.logits[..., :5])
cache = [[k_or_v[..., -5:, :] for k_or_v in kv] for kv in cache]
out = base_model(input_ids=segs[1]['input_ids'][:, 1:2], attention_mask=segs[1]['attention_mask'][..., 1:2, 1:], past_key_values=cache, use_cache=True, cache_position=torch.full((1,), 6))
print(out.logits[..., :5])

tensor([[[14.7198, 11.1084, 13.7085, 10.8592, 11.5626]]],
       grad_fn=<SliceBackward0>)
tensor([[[14.1674, 15.3581, 20.4713, 16.2741, 15.4210]]],
       grad_fn=<SliceBackward0>)


In [63]:
segs = model.segment(input_ids=input_ids, attention_mask=attention_mask)
cache = base_model(**segs[0], use_cache=True).past_key_values

attn0 = model.attn_mask_to_4d(attn_mask=segs[0]['attention_mask'], upper=True, query_len=2)
attn1 = model.attn_mask_to_4d(attn_mask=segs[1]['attention_mask'], upper=False, query_len=2)
# attn0[0][0][-1][0] = 1
segs[1]['attention_mask'] = torch.cat([attn0, attn1], dim=-1)
segs[1]['attention_mask'] = model.memory_cell.convert_to_infinity_attn_mask(segs[1]['attention_mask'], dtype=torch.bfloat16)
out = base_model(**segs[1], past_key_values=cache, use_cache=True, cache_position=torch.arange(2) + 5)
out.logits

tensor([[[14.7198, 11.1084, 13.7085,  ..., -0.2665, -0.2665, -0.2666],
         [14.1674, 15.3581, 20.4713,  ...,  0.1369,  0.1364,  0.1364]]],
       grad_fn=<UnsafeViewBackward0>)

In [42]:
segs[1]['attention_mask']

tensor([[[[-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,
           -0.0000e+00, -3.3895e+38],
          [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,
           -0.0000e+00, -0.0000e+00]]]])